# 推理显存基准测试 (Inference Memory Benchmark)

本笔记本复用 `alignmmbench_inference.ipynb` 的配置与推理管线，
额外提供「在不同 batch size 下测量推理显存峰值」的功能。

- **模型对象 / 检查点路径** 与 **处理器对象** 均由用户在第 1 节自行提供（与原笔记本一致）。
- 对 `BATCH_SIZES` 中的每个批大小调用一次 `run_inference`，并记录该批次推理期间的显存峰值。
- 推理超参数（`MAX_NEW_TOKENS` / `TEMPERATURE` / `TOP_P` / `SYSTEM_PROMPT` 等）完全沿用原笔记本设置。

| 节 | 内容 |
|---|---|
| 0 | 环境与路径设置 |
| 1 | 参数配置（模型 / 处理器由用户提供） |
| 2 | 显存基准测试循环 |
| 3 | 查看与可视化结果 |

## 0. 环境与路径设置

In [1]:
import sys
import os

print('Python executable:', sys.executable)
print('Python version   :', sys.version.split()[0])

NOTEBOOK_DIR = os.path.dirname(os.path.abspath('__file__'))
REPO_ROOT    = os.path.abspath(os.path.join(NOTEBOOK_DIR, '..'))
TRAIN_DIR    = os.path.join(REPO_ROOT, 'qwen3smvl', 'train')

for _p in [REPO_ROOT, TRAIN_DIR]:
    if _p not in sys.path:
        sys.path.insert(0, _p)

# ── 确保项目根目录在 sys.path 中，使 qwen3smvl 包可被导入 ──────────────────────
# 笔记本位于 <project_root>/notebooks/，需上移一级才能到达项目根目录
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(f'Project root     : {PROJECT_ROOT}')

try:
    import transformers, accelerate
    print(f'transformers     : {transformers.__version__}')
    print(f'accelerate       : {accelerate.__version__}')
    print('Import check     : OK')
    from transformers import (
    AutoProcessor,
    AutoModelForImageTextToText,
    AutoTokenizer,
    AutoModelForCausalLM,
)
    from train import load_processor, load_model_v2
except Exception as e:
    print('Import check     : FAIL →', type(e).__name__, e)
    print('\n→ Kernel 未使用项目 .venv，请在右上角内核选择器中切换为')
    print('  .venv\\Scripts\\python.exe')

Python executable: e:\cursorprojects\Qwen3-SmVL\.venv\Scripts\python.exe
Python version   : 3.12.13
Project root     : e:\cursorprojects\Qwen3-SmVL


e:\cursorprojects\Qwen3-SmVL\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
e:\cursorprojects\Qwen3-SmVL\.venv\Lib\site-packages\torch\cuda\__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


transformers     : 5.8.0
accelerate       : 1.10.1
Import check     : OK


In [2]:
import logging
import json
from datetime import datetime
from pathlib import Path

# ── 日志文件位置：可按需修改 LOG_DIR ──────────────────────────────────────────
LOG_DIR = Path(PROJECT_ROOT) / 'inference_log'
LOG_DIR.mkdir(parents=True, exist_ok=True)
LOG_FILE = LOG_DIR / f"memory_benchmark_{datetime.now():%Y-%m-%d_%H-%M-%S}.log"

# ── 日志配置：INFO 级别，方便观察推理进度，同时输出到控制台与文件 ─────────────
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(name)s - %(message)s',
    datefmt='%H:%M:%S',
    handlers=[
        logging.StreamHandler(),
        logging.FileHandler(LOG_FILE, mode='w', encoding='utf-8'),
    ],
)

print(f'日志将保存至: {LOG_FILE}')

# ── 导入核心推理函数 ────────────────────────────────────────────────────────
from qwen3smvl.inference.inference_vl_model import run_inference

print('run_inference 导入成功')

日志将保存至: e:\cursorprojects\Qwen3-SmVL\inference_log\memory_benchmark_2026-06-16_22-25-15.log
run_inference 导入成功


## 1. 参数配置（模型与处理器由用户提供）

与 `alignmmbench_inference.ipynb` 一致：`MODEL_OR_CHECKPOINT_PATH` 既可传入**已构造好的模型对象**，
也可传入**本地检查点目录路径**；`PROCESSOR` 传入处理器对象（或 `None` 让内部回退到 `load_processor()`）。

> ★ 建议传入**模型对象**：这样模型只加载一次，可在所有 batch size 间复用，显存测量更干净，
> 也避免每个 batch size 重新加载模型带来的耗时与显存抖动。若传入**路径**，`run_inference` 会在
> 每个 batch size 迭代内部重新构建并加载模型。

In [ ]:
import torch
DEVICE          = "cuda" if torch.cuda.is_available() else "cpu"



#用于处理数据的processor
# _PROC_PATH = os.path.join(PROJECT_ROOT, "model", "Qwen3.5-0.8B")
# PROCESSOR = AutoProcessor.from_pretrained(
#         _PROC_PATH
#     )

_SMOLVLM_PATH = os.path.join(PROJECT_ROOT, "model", "SmolVLM2-256M-Video-Instruct")
MODEL_OR_CHECKPOINT_PATH = AutoModelForImageTextToText.from_pretrained(
        _SMOLVLM_PATH,
        torch_dtype=torch.bfloat16,
        _attn_implementation="eager",
    ).to(DEVICE)

# ── 处理器对象（由用户提供）─────────────────────────────────────────────────
# 设为 None 时，run_inference 内部会回退到默认 load_processor()。
# PROCESSOR = load_processor()
PROCESSOR = AutoProcessor.from_pretrained(
    _SMOLVLM_PATH
)

# ── 模型对象 / 检查点路径（由用户提供）──────────────────────────────────────
# 可选检查点示例：
MODEL_OR_CHECKPOINT_PATH = load_model_v2(
    trained_model_path=os.path.join(PROJECT_ROOT, "model", "full_tuning"),
    device=DEVICE,
    new_vocab_size=len(PROCESSOR.tokenizer) if PROCESSOR is not None else None,
)
# MODEL_OR_CHECKPOINT_PATH = os.path.join(PROJECT_ROOT, "model", "unfreeze_connector_pretraining_full")

# M_PATH = os.path.join(PROJECT_ROOT, "model", "Qwen3.5-0.8B")
# MODEL_OR_CHECKPOINT_PATH = AutoModelForImageTextToText.from_pretrained(
#         M_PATH,
#         torch_dtype=torch.bfloat16,
#         _attn_implementation="eager",
#     ).to("cuda" if torch.cuda.is_available() else "cpu")

# ── AlignMMBench 数据集路径 ────────────────────────────────────────────────
JSONL_PATH  = os.path.join(PROJECT_ROOT, "data", "AlignMMBench", "metadata.jsonl")
IMAGE_ROOT  = os.path.join(PROJECT_ROOT, "data", "AlignMMBench")

# ── 输出目录（每个 batch size 的预测结果分别写入独立文件）──────────────────
OUTPUT_DIR  = os.path.join(PROJECT_ROOT, "inference_memory_estimation")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── 连接器形状日志：None 表示禁用 CSV 记录 ─────────────────────────────────
CSV_PATH = None

# ── 推理超参数（沿用 alignmmbench_inference.ipynb 的设置）───────────────────
MAX_NEW_TOKENS  = 256     # 模型单次最多生成的 token 数（越大 KV cache 显存峰值越高）
TEMPERATURE     = 0.8      # 采样温度
TOP_P           = 0.95     # top-p 核采样参数
SYSTEM_PROMPT   = "你是一个有帮助的语言与视觉助手。你能够理解用户提供的视觉内容，并使用自然语言协助用户完成各种任务。"
ENABLE_THINKING = False    # 是否启用 <think>…</think> 思考模式
ADD_MEDIA_INTRO_OUTRO = True  # 与训练 collator 同名参数对齐

# ── 显存基准测试专用参数 ───────────────────────────────────────────────────
# 待测试的 batch size 列表（按需增删；显存够大时可继续往上加）
BATCH_SIZES = [1]
# 每个 batch size 实际跑多少个完整批次（仅在 MAX_SAMPLES_PER_TEST 为 None 时生效，
# 此时样本数 = batch_size * N_BATCHES_PER_SIZE）。
# 取 >=1 即可；取 2 可让显存峰值测量更稳定（覆盖多批序列长度差异）。
N_BATCHES_PER_SIZE = 2
# 每次测试使用样本数的「上限」，对基准值 batch_size * N_BATCHES_PER_SIZE 封顶：
#   - None  → 不封顶，直接用 batch_size * N_BATCHES_PER_SIZE；
#   - 整数 N → n_samples = min(batch_size * N_BATCHES_PER_SIZE, N)，
#             即最多用 N 个样本；当基准值更小时，多出来的样本闲置不使用。
#             注意：若 N < batch_size，该 size 只会跑一个不完整批次，
#             显存峰值不能代表真正的 batch_size 规模，请按需取 N >= max(BATCH_SIZES)。
# 数据集总样本数有限时，run_inference 内部还会进一步截断到可用上限。
MAX_SAMPLES_PER_TEST = 10

_model_label = (MODEL_OR_CHECKPOINT_PATH
                if isinstance(MODEL_OR_CHECKPOINT_PATH, (str, os.PathLike))
                else f"<{type(MODEL_OR_CHECKPOINT_PATH).__name__} instance>")
print(f"设备            : {DEVICE}")
print(f"模型/检查点      : {_model_label}")
print(f"处理器          : {type(PROCESSOR).__name__ if PROCESSOR is not None else 'None (默认 load_processor())'}")
print(f"数据集 JSONL    : {JSONL_PATH}")
print(f"图像根目录      : {IMAGE_ROOT}")
print(f"待测 batch size : {BATCH_SIZES}")
print(f"每个 size 批次数 : {N_BATCHES_PER_SIZE}")
print(f"样本数上限       : {MAX_SAMPLES_PER_TEST if MAX_SAMPLES_PER_TEST is not None else '无 (= batch_size * N_BATCHES_PER_SIZE)'}")

assert os.path.exists(JSONL_PATH), f"JSONL 文件不存在: {JSONL_PATH}"
assert os.path.exists(IMAGE_ROOT), f"图像根目录不存在: {IMAGE_ROOT}"
print("\n路径检查通过 ✓")

00:07:54 [INFO] qwen3smvl.utils - 正在加载SmolVLM2处理器...
00:07:54 [INFO] qwen3smvl.utils - 正在加载Qwen3分词器...
00:07:55 [INFO] qwen3smvl.utils -   添加前词表大小: 151669
00:07:55 [INFO] qwen3smvl.utils -   已新增 36 个位置特殊token（<row_i_col_j>）
00:07:55 [INFO] qwen3smvl.utils -   添加后词表大小: 151705
00:07:55 [INFO] qwen3smvl.utils - 正在配置处理器...
00:07:55 [INFO] qwen3smvl.utils -   图像处理器（来自 preprocessor_config.json）: size=SizeDict(height=None, width=None, longest_edge=2048, shortest_edge=None, max_height=None, max_width=None), max_image_size={'longest_edge': 512}, do_image_splitting=True
00:07:55 [INFO] qwen3smvl.utils - 正在加载SmolVLM2视觉-语言模型...
Loading weights: 100%|██████████| 471/471 [00:00<00:00, 3346.96it/s]
00:07:56 [INFO] qwen3smvl.utils - 正在加载Qwen3语言模型...
Loading weights: 100%|██████████| 311/311 [00:00<00:00, 10835.11it/s]
00:07:58 [INFO] qwen3smvl.utils - 正在构建连接器配置...
00:07:58 [INFO] qwen3smvl.utils - 正在创建新的连接器...
00:07:58 [INFO] qwen3smvl.utils - new_vocab_size=151705 ≤ 实际嵌入矩阵大小 151936，新token已落入padding行，

设备            : cuda
模型/检查点      : <SmolVLMForConditionalGeneration instance>
处理器          : SmolVLMProcessor
数据集 JSONL    : e:\cursorprojects\Qwen3-SmVL\data\AlignMMBench\metadata.jsonl
图像根目录      : e:\cursorprojects\Qwen3-SmVL\data\AlignMMBench
待测 batch size : [1]
每个 size 批次数 : 2
样本数上限       : 10

路径检查通过 ✓


## 2. 显存基准测试循环

对每个 `batch_size`：
1. `gc.collect()` + `empty_cache()` + `reset_peak_memory_stats()`，并记录推理前的基线显存；
2. 调用 `run_inference`（样本数 = `min(batch_size * N_BATCHES_PER_SIZE, MAX_SAMPLES_PER_TEST)`）；
3. 读取 `max_memory_reserved` / `max_memory_allocated` 作为该批次的显存峰值；
4. 捕获 CUDA OOM：记为 `OOM` 并继续测试下一个 batch size；
5. 每个 batch size 完成后，把累计的显存统计**增量写入** `smolvlm2_raw_infer_memory_summary.csv` / `.json`（中途中断也能保留已完成结果）。

> 说明：`run_inference` 内部会在**模型加载之后、推理循环之前**再次 `reset_peak_memory_stats`，
> 因此 `peak_reserved_GB` 反映的是「推理阶段」峰值；`base_reserved_GB` 为推理前基线（含已驻留的模型权重），
> `peak_inference_GB = peak - base` ≈ KV cache + 激活的增量显存。

In [ ]:
import gc
import time
import pandas as pd

# ── 解析显存统计所用的设备索引（与 run_inference 内部逻辑保持一致）──────────
_use_cuda = torch.cuda.is_available() and not str(DEVICE).startswith("cpu")
if _use_cuda:
    _device_idx = torch.device(DEVICE).index
    if _device_idx is None:
        _device_idx = torch.cuda.current_device()
    _props = torch.cuda.get_device_properties(_device_idx)
    _total_mem_gb = round(_props.total_memory / 1024 ** 3, 3)
    print(f"GPU = {_props.name}，总显存 = {_total_mem_gb} GB")
else:
    _device_idx = None
    _total_mem_gb = None
    print("⚠️ 未检测到可用 CUDA，显存峰值列将为 None（仍会执行推理）。")


def _gb(n_bytes):
    return round(n_bytes / 1024 ** 3, 3)


records = []

mdl_name = "qwen3smvl_full"
# ── 显存统计输出文件 ────────────────────────────────────────────────────────
# 在循环内增量写入：即使中途 OOM / 手动中断，已完成的 batch size 结果也会落盘。
# MEM_STATS_CSV  = os.path.join(OUTPUT_DIR, f"qwen3smvl_full_infer_memory_summary_{BATCH_SIZES[0]}.csv")
MEM_STATS_JSON = os.path.join(OUTPUT_DIR, f"{mdl_name}_infer_memory_summary_{BATCH_SIZES[0]}.json")


def _save_mem_stats(rows):
    """把当前已收集的显存统计写入 CSV 与 JSON（每个 batch size 完成后调用一次）。"""
    df = pd.DataFrame(rows)
    df.to_csv(MEM_STATS_CSV, index=False, encoding="utf-8-sig")
    df.to_json(MEM_STATS_JSON, orient="records", indent=2, force_ascii=False)
    return df


for bs in BATCH_SIZES:
    # 该 batch size 测试使用的输入样本数：
    #   - 基准值 = bs * N_BATCHES_PER_SIZE（跑满 N_BATCHES_PER_SIZE 个完整批次）；
    #   - 若设置了 MAX_SAMPLES_PER_TEST，则对基准值封顶（取较小者），
    #     基准值更小时多出来的样本闲置不使用。
    n_samples = bs * N_BATCHES_PER_SIZE
    if MAX_SAMPLES_PER_TEST is not None:
        n_samples = min(n_samples, MAX_SAMPLES_PER_TEST)
    out_path = os.path.join(OUTPUT_DIR, f"memory_benchmark_bs{bs}.jsonl")

    base_reserved = peak_reserved = peak_allocated = peak_inference = None
    status = "ok"

    # 清理并复位峰值计数器，使本次测量只反映该 batch size 的推理。
    if _use_cuda:
        gc.collect()
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats(_device_idx)
        base_reserved = _gb(torch.cuda.memory_reserved(_device_idx))

    print(f"\n{'=' * 64}\n▶ batch_size={bs}  (num_samples={n_samples})\n{'=' * 64}")
    t0 = time.time()
    try:
        run_inference(
            model_or_checkpoint_path=MODEL_OR_CHECKPOINT_PATH,
            processor=PROCESSOR,
            output_path=out_path,
            jsonl_path=JSONL_PATH,
            image_root=IMAGE_ROOT,
            csv_path=CSV_PATH,
            batch_size=bs,
            max_new_tokens=MAX_NEW_TOKENS,
            temperature=TEMPERATURE,
            top_p=TOP_P,
            system_prompt=SYSTEM_PROMPT,
            enable_thinking=ENABLE_THINKING,
            add_media_intro_outro=ADD_MEDIA_INTRO_OUTRO,
            max_samples=n_samples,
            device=DEVICE,
        )
    except RuntimeError as e:
        # 捕获 CUDA OOM（torch.cuda.OutOfMemoryError 是 RuntimeError 的子类）。
        if "out of memory" in str(e).lower():
            status = "OOM"
            print(f"❌ batch_size={bs} 显存溢出 (OOM)：{e}")
            if _use_cuda:
                torch.cuda.empty_cache()
        else:
            raise

    elapsed_s = round(time.time() - t0, 2)

    if _use_cuda:
        peak_reserved = _gb(torch.cuda.max_memory_reserved(_device_idx))
        peak_allocated = _gb(torch.cuda.max_memory_allocated(_device_idx))
        if base_reserved is not None:
            peak_inference = round(max(0.0, peak_reserved - base_reserved), 3)

    records.append({
        "batch_size": bs,
        "num_samples": n_samples,
        "status": status,
        "elapsed_s": elapsed_s,
        "base_reserved_GB": base_reserved,
        "peak_reserved_GB": peak_reserved,
        "peak_allocated_GB": peak_allocated,
        "peak_inference_GB": peak_inference,
        "peak_reserved_pct": (round(peak_reserved / _total_mem_gb * 100, 2)
                              if (_use_cuda and peak_reserved is not None) else None),
    })

    # 该 batch size 完成后立即落盘，保证中途中断也能保留已有结果。
    _save_mem_stats(records)

    if _use_cuda:
        gc.collect()
        torch.cuda.empty_cache()

mem_df = _save_mem_stats(records)
print("\n显存基准测试完成。")
print(f"显存统计已保存至:\n  - {MEM_STATS_CSV}\n  - {MEM_STATS_JSON}")
mem_df

00:08:07 [INFO] qwen3smvl.inference.inference_vl_model - Loading dataset from e:\cursorprojects\Qwen3-SmVL\data\AlignMMBench\metadata.jsonl...
00:08:07 [INFO] qwen3smvl.inference.inference_vl_model - Loaded 2 samples
00:08:07 [INFO] qwen3smvl.inference.inference_vl_model - 使用调用方提供的自定义 processor（type=SmolVLMProcessor），跳过 load_processor()
00:08:07 [INFO] qwen3smvl.inference.inference_vl_model - 接收到预构建模型实例（type=SmolVLMForConditionalGeneration），跳过 load_model_v2()；strict / 权重探测 / 词表 resize 等仅在路径分支生效。
00:08:07 [INFO] qwen3smvl.inference.inference_vl_model - 
Running inference on 2 samples with batch_size=1 for model <SmolVLMForConditionalGeneration instance>...
00:08:07 [INFO] qwen3smvl.inference.inference_vl_model - GPU = NVIDIA GeForce RTX 5090 Laptop GPU. Max memory = 23.889 GB.
00:08:07 [INFO] qwen3smvl.inference.inference_vl_model - 1.336 GB of memory reserved.
00:08:07 [INFO] qwen3smvl.inference.inference_vl_model - 开始推理...


GPU = NVIDIA GeForce RTX 5090 Laptop GPU，总显存 = 23.889 GB

▶ batch_size=1  (num_samples=2)


Inference:   0%|          | 0/2 [00:00<?, ?it/s]00:08:07 [INFO] qwen3smvl.inference.inference_vl_model - [{'question_id': '00000000-0', 'images': <PIL.Image.Image image mode=RGB size=499x469 at 0x20E54BEBE60>, 'texts': '描述图片。', 'history': [], 'image_path': 'e:\\cursorprojects\\Qwen3-SmVL\\data\\AlignMMBench\\images/000000000.jpg'}]
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
00:08:07 [INFO] qwen3smvl.inference.inference_vl_model - Processed text: <|im_start|>system
你是一个有帮助的语言与视觉助手。你能够理解用户提供的视觉内容，并使用自然语言协助用户完成各种任务。<|im_end|>
<|im_start|>user
以下是一些图片：<|image_pad|>
现在请回答以下问题或者完成以下要求：描述图片。<|im_end|>
<|im_start|>assistant
<think>

</think>


00:08:07 [INFO] qwen3smvl.inference.inference_vl_model - Batch image row shape before model processor:  1
00:08:07 [INFO] qwen3smvl.inference.inference_vl_model - total number of images in batch: 1
00:08:07 [INFO] qwen3smvl.inference.inference_vl_model - Batch image tensor shape after mod


显存基准测试完成。
显存统计已保存至:
  - e:\cursorprojects\Qwen3-SmVL\inference_memory_estimation\smolvlm2_raw_infer_memory_summary_1.csv
  - e:\cursorprojects\Qwen3-SmVL\inference_memory_estimation\qwen3smvl_full_infer_memory_summary_1.json


,batch_size,num_samples,status,elapsed_s,base_reserved_GB,peak_reserved_GB,peak_allocated_GB,peak_inference_GB,peak_reserved_pct
0,1,2,ok,26.22,1.336,4.057,3.573,2.721,16.98


: 

## 3. 查看与可视化结果

绘制不同 batch size 下的显存峰值曲线，并把汇总表保存为 CSV。

In [ ]:
import matplotlib.pyplot as plt

# 仅对成功（非 OOM）且有显存统计的行作图
plot_df = mem_df[(mem_df["status"] == "ok") & mem_df["peak_reserved_GB"].notna()]

if len(plot_df) == 0:
    print("没有可绘制的显存数据（可能未启用 CUDA，或所有 batch size 均 OOM）。")
else:
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(plot_df["batch_size"], plot_df["peak_reserved_GB"],
            marker="o", label="peak reserved")
    ax.plot(plot_df["batch_size"], plot_df["peak_allocated_GB"],
            marker="s", label="peak allocated")
    ax.plot(plot_df["batch_size"], plot_df["peak_inference_GB"],
            marker="^", label="inference delta (peak - base)")
    if _total_mem_gb is not None:
        ax.axhline(_total_mem_gb, color="red", linestyle="--",
                   label=f"GPU total {_total_mem_gb} GB")
    ax.set_xlabel("batch size")
    ax.set_ylabel("memory (GB)")
    ax.set_title("Inference peak GPU memory vs batch size")
    ax.set_xticks(plot_df["batch_size"])
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

# 显存统计已在第 2 节的基准测试循环中增量写入（CSV + JSON），此处仅提示文件位置。
print(f"显存统计文件:\n  - {MEM_STATS_CSV}\n  - {MEM_STATS_JSON}")